In [ ]:
using MomentumED
using CUDA
MomentumED.Methods._throw_cuda_unavailable() # throw an error if CUDA is not available

In [ ]:
# preparation for 1/3 filling with 12 electrons in 36 orbitals
# subspace momentum decomposition will cost many minutes, althrough accelerated with multiple CPU threads
# will need 10GB VRAM if using krylov_dim = 20 (default value is 30)

# using MKL, MKLSparse
# MKL.BLAS.set_num_threads(1)
using LinearAlgebra
include("../Landau level torus.jl")
using .LLT
k_list = [0 1 2 3 4 5 0 1 2 3 4 5 0 1 2 3 4 5 0 1 2 3 4 5 0 1 2 3 4 5 0 1 2 3 4 5;
          0 0 0 0 0 0 1 1 1 1 1 1 2 2 2 2 2 2 3 3 3 3 3 3 4 4 4 4 4 4 5 5 5 5 5 5]
Nk = 36
Gk = (6, 6)
Ne = 12
sys_int = LandauInteraction(tri_lattice, (1, 0, 1, 0))
sys_int.D_l = 10.0
sys_int.mix = 0          # mix * Haldane + (1-mix) * Coulomb
para = EDPara(k_list, Gk, H_two = sys_int);
@time subspaces, ss_k = ED_momentum_subspaces(para, Ne);
display(collect(zip(length.(subspaces), ss_k)))
scat = ED_scatterlist_twobody(para);
hmlt = MBOperator(scat, upper_hermitian = true)
# when method = :map or :cuda_map, EDsolve only accept Hamiltonian operator and refuses scatter list.

In [ ]:
MomentumED.release_cuda()        # clean up GPU memory
# output the gpu memory at the peaks, default to be true
MomentumED.CUDA_MEMORY_MONITOR[] = true

In [ ]:
# the 3 ground states are in the same momentum subspace (0,0)
Neigen = 9  # Number of eigenvalues to compute per subspace
ss_range = 1:1
# ss_range = 1:36
energies = [Vector{Float64}() for _ in ss_range];
vectors = Vector{Vector{<:MBS64Vector}}(undef, length(ss_range));
for i in eachindex(ss_range)
    println("Processing subspace #$i with size $(length(subspaces[ss_range[i]])), momentum $(ss_k[i])")
    energies[i], vectors[i] = EDsolve(subspaces[ss_range[i]], hmlt;
        N = Neigen, showtime = true, ishermitian = true,
        # method = :map,                 # linear map method in cpu (will be slow for large size)
        method = :cuda_map,            # linear map method in CUDA gpu
        krylov_dim = 20,               # dimension of the krylov space, determines the memory usage
        verbosity = 4,                 # show krylov-schur algorithm details
    )
end

In [ ]:
using CairoMakie
function plot_ed_spectrum(energies, ss_k, Gk::NTuple{2, Int64};
    title = nothing, ylims = (nothing, nothing),
    ylabel = "Energy per unit cell (W₀ = e²/ϵl)",
    top_subspace_number = true)

    fig = Figure();
    ax = Axis(fig[1, 1];
        xlabel = "k1+$(Gk[1])k2",
        ylabel = ylabel
    )
    ax_top = Axis(fig[1, 1];
        xaxisposition = :top
    )
    top_ticks = ([], [])
    hidespines!(ax_top)
    hidexdecorations!(ax_top; label = false, ticklabels = false)
    hideydecorations!(ax_top)
    linkxaxes!(ax, ax_top)

    # Plot energy levels for each momentum block
    for i in eachindex(ss_k)
        x = ss_k[i][1] + Gk[1] * ss_k[i][2]
        push!(top_ticks[1], x)
        push!(top_ticks[2], string(i))
        if isassigned(energies,i)
            for e in energies[i]
                scatter!(ax, x, e, color = :blue, marker=:hline)
            end
        end
    end
    ylims!(ax, ylims...)
    top_subspace_number && (ax_top.xticks = top_ticks)
    if title isa String
        ax_top.subtitle = title
    end
    display(fig)
    fig, ax
end

In [ ]:
plot_ed_spectrum(energies/Nk/LLT.W0, ss_k, Gk;
    title = "Nk = $Nk, Ne = $Ne",
);